# Validate one RTSTRUCT ROI and reference mesh

This notebook only visualizes artifacts produced by `scripts/rtstruct_mesh.py build`. All DICOM parsing, rasterization, mesh extraction, and round-trip validation live in reusable Python modules.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from ipywidgets import interact, IntSlider

from medgs4d.mesh_validation import load_case, mask_boundary, window_hu


## Configuration

In [ ]:
# Change only this path.
CASE_DIR = Path(
    "/home/jovyan/shared/mtm_medgs_stack/data/tcia_4d_lung/prepared/"
    "patient117_20001024_annotated/rtstruct_mesh/phase_00"
)

case = load_case(CASE_DIR)
manifest = case["manifest"]
geometry = case["geometry"]
ct = case["ct_volume"]
mask = case["mask"]
roundtrip = case["roundtrip_mask"]
mesh = case["mesh"]
contours = case["contours"]
report = case["report"]

print(f"ROI: {manifest['roi_name']}")
print(f"Phase: {manifest['phase_percent']:g}%")
print(f"CT shape: {ct.shape}")
print(f"Spacing z,y,x [mm]: {geometry.spacing_zyx}")


## Numerical validation report

In [ ]:
display(pd.DataFrame({"Value": report}).rename_axis("Metric"))
display(pd.read_csv(CASE_DIR / manifest["contour_report_file"]).head(20))


## Axial overlay

Solid boundary: rasterized RTSTRUCT mask. Dashed boundary: mask reconstructed from the mesh. Original RTSTRUCT contour points are overlaid as lines.

In [ ]:
contours_by_slice = {}
for contour in contours:
    contours_by_slice.setdefault(int(contour["slice_index"]), []).append(np.asarray(contour["points_zyx"]))

areas = mask.reshape(mask.shape[0], -1).sum(axis=1)
default_slice = int(np.argmax(areas))

def show_axial(slice_index: int):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(window_hu(ct[slice_index]), cmap="gray")
    if mask[slice_index].any():
        ax.contour(mask[slice_index], levels=[0.5], linewidths=2)
    if roundtrip[slice_index].any():
        ax.contour(roundtrip[slice_index], levels=[0.5], linewidths=1.5, linestyles="--")
    for points_zyx in contours_by_slice.get(slice_index, []):
        points_xy = points_zyx[:, [2, 1]]
        points_xy = np.vstack([points_xy, points_xy[0]])
        ax.plot(points_xy[:, 0], points_xy[:, 1], linewidth=1)
    ax.set_title(f"Axial slice {slice_index}")
    ax.set_axis_off()
    plt.show()

interact(
    show_axial,
    slice_index=IntSlider(
        value=default_slice, min=0, max=ct.shape[0] - 1, step=1, continuous_update=False
    ),
)


## Orthogonal mask views

In [ ]:
occupied = np.argwhere(mask)
z_mid, y_mid, x_mid = np.round(occupied.mean(axis=0)).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
axes[0].imshow(window_hu(ct[z_mid]), cmap="gray")
axes[0].contour(mask[z_mid], levels=[0.5])
axes[0].set_title(f"Axial z={z_mid}")

axes[1].imshow(window_hu(ct[:, y_mid, :]), cmap="gray", aspect="auto")
axes[1].contour(mask[:, y_mid, :], levels=[0.5])
axes[1].set_title(f"Coronal y={y_mid}")

axes[2].imshow(window_hu(ct[:, :, x_mid]), cmap="gray", aspect="auto")
axes[2].contour(mask[:, :, x_mid], levels=[0.5])
axes[2].set_title(f"Sagittal x={x_mid}")

for ax in axes:
    ax.set_axis_off()
plt.show()


## Interactive 3D mesh in patient coordinates [mm]

In [ ]:
vertices = mesh.vertices_xyz
faces = mesh.faces
figure = go.Figure(
    data=[
        go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            opacity=0.75, flatshading=False,
        )
    ]
)
figure.update_layout(
    title=f"{manifest['roi_name']} — phase {manifest['phase_percent']:g}%",
    scene=dict(
        xaxis_title="Patient x [mm]",
        yaxis_title="Patient y [mm]",
        zaxis_title="Patient z [mm]",
        aspectmode="data",
    ),
    width=900, height=750,
)
figure.show()


## Round-trip mismatch

In [ ]:
difference = np.logical_xor(mask, roundtrip)
difference_areas = difference.reshape(difference.shape[0], -1).sum(axis=1)
worst_slice = int(np.argmax(difference_areas))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
axes[0].imshow(mask[worst_slice], cmap="gray")
axes[0].set_title("Source mask")
axes[1].imshow(roundtrip[worst_slice], cmap="gray")
axes[1].set_title("Mesh → mask")
axes[2].imshow(difference[worst_slice], cmap="gray")
axes[2].set_title(f"XOR mismatch, slice {worst_slice}")
for ax in axes:
    ax.set_axis_off()
plt.show()


## Acceptance checklist

- The original RTSTRUCT contour follows the intended anatomy.
- The rasterized mask boundary overlaps the original contour.
- Axial, coronal, and sagittal views have the expected orientation.
- The 3D mesh has plausible scale and orientation in millimetres.
- `mesh_watertight` is true.
- Round-trip Dice is above the configured threshold.
- No isolated or implausibly distant components are visible.